# LeadLens pipeline walkthrough

Runs the same code the API uses, directly on the bundled SaaSquatch-style export, so you can see each stage: **clean → dedupe → score → Web Intel → Exit-Readiness → Credit Optimizer**. No server needed.

Setup (from the repo root, after creating the backend virtualenv):

```
cd backend
.venv/Scripts/python -m pip install -r ../notebooks/requirements.txt
.venv/Scripts/python -m ipykernel install --user --name leadlens --display-name "LeadLens (backend venv)"
.venv/Scripts/python -m jupyter notebook ../notebooks/walkthrough.ipynb
```

Pick the **LeadLens (backend venv)** kernel, then Run All.

In [ ]:
import sys, os, asyncio
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend').is_dir() and (p / 'data').is_dir())
sys.path.insert(0, str(ROOT / 'backend'))
os.chdir(ROOT / 'backend')
import pandas as pd
from app.pipeline.normalize import map_columns, normalize_row, quality_score
from app.pipeline.dedupe import find_clusters, merge_cluster
from app.pipeline.score import fit_score, exit_score, quadrant_for
from app.pipeline.fetcher import fetch_site
from app.pipeline.extract import extract
from app.llm.rules import RulesProvider

df = pd.read_csv(ROOT / 'data' / 'saasquatch_sample.csv', dtype=str, keep_default_na=False)
print(df.shape); df.head(5)

## 1. Clean: bad values become `None` + a flag, never a guess

In [ ]:
colmap = map_columns(list(df.columns))
rows, flags = [], []
for rec in df.to_dict(orient='records'):
    r, f = normalize_row(rec, colmap); rows.append(r); flags.append(f)
from collections import Counter
Counter(f for fl in flags for f in fl).most_common(12)

In [ ]:
# examples of the 'revenue = 4' problem seen in the SaaSquatch demo
bad = [(df.iloc[i]['Company'], df.iloc[i]['Revenue'], rows[i]['revenue'], flags[i]) for i in range(len(rows)) if 'revenue_suspect' in flags[i]]
bad[:5]

## 2. Dedupe: fuzzy clusters with provenance

In [ ]:
clusters = find_clusters(rows)
multi = [c for c in clusters if len(c) > 1]
print(f'{len(rows)} rows -> {len(clusters)} companies; {len(multi)} clusters had duplicates')
biggest = max(multi, key=len)
[(df.iloc[i]['Company'], df.iloc[i]['Revenue'], df.iloc[i]['Employees Count']) for i in biggest]

In [ ]:
merged, mflags, conflicts = merge_cluster(rows, flags, biggest, list(range(len(rows))))
print({k: merged[k] for k in ('company','revenue','employees','email','phone')})
conflicts

## 3. Score every company (rules only at this point)

In [ ]:
leads = []
for c in clusters:
    m, fl, _ = merge_cluster(rows, flags, c, list(range(len(rows))))
    m['revenue_usd'] = m.pop('revenue'); m['quality_score'] = quality_score({**m, 'revenue': m['revenue_usd']}, fl)
    m['fit_score'], m['fit_grade'], m['fit_breakdown'] = fit_score(m)
    m['exit_score'], m['exit_breakdown'] = exit_score(m, None)
    m['quadrant'] = quadrant_for(m['fit_score'], m['exit_score'])
    leads.append(m)
L = pd.DataFrame(leads)
print(L.quadrant.value_counts()); print(L.fit_grade.value_counts())
L.sort_values(['fit_score','exit_score'], ascending=False)[['company','industry','revenue_usd','employees','year_founded','fit_score','exit_score','quadrant']].head(10)

## 4. Web Intel on one company (fixture mode: bundled offline site)

In [ ]:
ranked = L[L.domain.notna()].sort_values(['fit_score', 'exit_score'], ascending=False)
for _, cand in ranked.iterrows():          # first top lead whose (bundled) website is reachable
    top = cand.to_dict(); res = await fetch_site(top['domain'])
    if res.status == 'ok': break
ex = extract(res.pages) if res.pages else {}
ex.update(wayback_first_year=res.wayback_first_year, wayback_last_year=res.wayback_last_year, mx_ok=res.mx_ok)
print(top['company'], '|', res.status)
{k: v for k, v in ex.items() if k not in ('text_excerpt',)}

In [ ]:
llm = await RulesProvider().analyze(top, ex)   # swap for GeminiProvider() with a key
before, _ = exit_score(top, None)
after, bd = exit_score(top, {'extracted': ex, 'llm': llm})
print(f'Exit-Readiness {before} -> {after}')
pd.DataFrame(bd)[['factor','points','max','reason']]

In [ ]:
print(llm['analyst_note']); print(); print(llm['outreach_angle'])

## 5. Credit Optimizer: 50 credits, where do they go?

In [ ]:
from app.models import Lead
from app.services.optimizer import optimize
objs = [Lead(dataset_id=0, **{k: v for k, v in m.items() if k in Lead.model_fields}) for m in leads]
for o, m in zip(objs, leads): o.id = leads.index(m)
plan = optimize(objs, 50)
print(plan['summary'])
pd.DataFrame(plan['picks']).head(10)